# Reproducing "Next word probability and similarity" — SOLUTIONS

✅ **Solutions notebook** — the completed version of
`next-word-probability-similarity.ipynb`. Give it to students only **after** they have
tried the exercises.
This notebook reproduces the decoding pipeline of the deck **Next word probability and
similarity** with the model behind it: **Qwen3-1.7B**. Three exercises:

1. the greedy decoding loop;
2. how a logit is actually calculated — a dot product, written by hand;
3. the same calculation applied after every transformer layer (the deck's
   "France pulls ahead" lens figure).

**Requirements**: `pip install torch transformers`. The model is 1.7B parameters: ~4 GB of
RAM in float32. **A CPU is enough** — each forward pass takes a few seconds.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"   # a CPU is enough
model = model.to(device).eval()   # eval(): inference only, no training

print("model loaded on", device)


### Exercise 1 — Greedy decoding, without `.generate()`

The deck's « Next-token probability from Transformer stacks » slide shows the loop:
embed → L transformer blocks → unembed → **argmax** → append → start again, until a
special `<|endoftext|>` token.

Implement `greedy_generate(prompt_text, max_new_tokens)` **by hand** (do **not** call
`model.generate`):

1. tokenize the prompt;
2. forward pass, take the logits of the last position;
3. `argmax` → next token id;
4. stop if it is the end-of-text token (`tokenizer.eos_token_id`), else append it to the ids;
5. repeat, and return the full decoded text.

**Expected (deck)**: `"Paris is the capital"` → `"␣of"`, then `"␣France"`, then
end-of-text. Final text: `"Paris is the capital of France"` (modulo the leading space of
the first generated token).

Note: the deck draws `<|endoftext|>` with id 100257 — an illustration borrowed from
another tokenizer. Print the **real** id for this model.


In [ ]:
# SOLUTION
def greedy_generate(prompt_text, max_new_tokens=10):
    ids = tokenizer(prompt_text, return_tensors="pt").input_ids.to(device)
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(input_ids=ids).logits[0, -1, :]
        next_id = torch.argmax(logits).item()        # greedy: most likely token
        if next_id == tokenizer.eos_token_id:
            break                                    # <|endoftext|>: stop
        ids = torch.cat([ids, torch.tensor([[next_id]], device=ids.device)], dim=1)
    return tokenizer.decode(ids[0])

print(greedy_generate("Paris is the capital"))
print("eos token id:", tokenizer.eos_token_id,
      "->", repr(tokenizer.decode([tokenizer.eos_token_id])))
# Expected (deck): argmax gives ' of' (315), then ' France' (9822), then <|endoftext|>
# -> "Paris is the capital of France". The deck draws <|endoftext|> with id 100257,
# borrowed from another tokenizer as an illustration; the real Qwen3 id is 151643.


### Exercise 2 — How a logit is calculated

The deck's « Next-token probability from Transformer stacks » slide says:

> similarity[4, 0] = row 4 of h_L · row 0 of W_U

The logit of a token is the **dot product** of the last hidden vector h_L with that
token's row of the unembedding matrix W_U

**Implement `logit_of(h, row)` in the second cell**

The third cell checks you: the logit of `␣France` computed by hand must match the
model's, and `W_U @ h` — nothing more than **V dot products**, one per row — must
reproduce the model's full logits vector, argmax included.

**Expected**: both checks pass — the logit *is* the dot product. Note the deck's remark:
a cosine would give the same ranking but would need all V = 151 936 rows normalized at
every decoding step, so the raw dot product wins on cost.


In [ ]:
lens_prompt = "The city of Paris is the capital of"
inputs_lens = tokenizer(lens_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out_lens = model(**inputs_lens, output_hidden_states=True)

hidden_states = out_lens.hidden_states
n_layers = model.config.num_hidden_layers
print(f"{n_layers} layers -> {len(hidden_states)} hidden states "
      "(index 0 = input embeddings)")

final_norm = model.model.norm        # final RMSNorm, applied before unembedding
W_U = model.lm_head.weight.detach()  # (V, d) — tied to the embeddings in this model

# the last hidden state is the output of layer 28, BEFORE the final norm
h = final_norm(hidden_states[-1][0, -1])
france_id = tokenizer.encode(" France", add_special_tokens=False)[0]


In [ ]:
# SOLUTION — multiply, then add, applied to two vectors of d = 2048 numbers
def logit_of(h, row):
    return (h * row).sum().item()


In [ ]:
# the logit of ' France', computed by hand, must match the model's
manual_logit = logit_of(h, W_U[france_id])
model_logit = out_lens.logits[0, -1, france_id].item()
print(f"manual logit(' France') = {manual_logit:.4f}")
print(f"model  logit(' France') = {model_logit:.4f}")
assert torch.isclose(torch.tensor(manual_logit), torch.tensor(model_logit), rtol=1e-4)



In [ ]:
# W_U @ h is nothing more than V dot products — one per row of W_U. Its argmax is
# the token greedy decoding would pick:
manual_logits = W_U @ h
assert (manual_logits - out_lens.logits[0, -1, :]).abs().max().item() < 1e-3
assert manual_logits.argmax().item() == out_lens.logits[0, -1, :].argmax().item()
print("top-1 token:", repr(tokenizer.decode([manual_logits.argmax().item()])))

# The logit IS the dot product: an unnormalized similarity between the last hidden
# state and the token's unembedding row. A cosine would require normalizing all
# V = 151 936 rows at every decoding step — far too expensive (deck, decoding slide).


### Exercise 3 — Layer by layer, « France » pulls ahead

The deck's lens figure takes the prompt **« The city of Paris is the capital of »**, and
after **each** of the 28 transformer layers, unembeds the last vector and shows the
top-1 token: the model converges city → country → France.

Loop over the 29 hidden states of exercise 2. For each: take the last position, apply
the final norm, compute the logits **with your own dot products** (`W_U @ h_last` — the
check from exercise 2, now applied to every layer), take the argmax, decode it. Print
`hidden state → top-1 token`.

Bonus: at which layer does `␣France` become the top-1 prediction? Does it stay there?

**Expected (deck)**: generic continuations in the early layers; `␣France` pulls ahead in
the later layers and stays.


In [ ]:
# SOLUTION — reuses hidden_states, final_norm and W_U from exercise 2
tops = []
for layer, h_layer in enumerate(hidden_states):
    h_last = final_norm(h_layer[0, -1])    # last position, normalized
    logits = W_U @ h_last                  # V dot products, as checked in exercise 2
    top_id = logits.argmax().item()
    top = tokenizer.decode([top_id])
    tops.append(top)
    print(f"hidden state {layer:>2} -> {top!r}")

# Bonus: when does ' France' take the lead, and does it keep it?
if " France" in tops:
    first = tops.index(" France")
    keeps = all(t == " France" for t in tops[first:])
    print(f"' France' first becomes top-1 after layer {first - 1} "
          f"and keeps the lead: {keeps}")
else:
    print("' France' never becomes top-1 — check the prompt and the final norm")
# Expected (deck, lens slide): generic continuations early, ' France' pulls ahead in
# the later layers and stays.


## Wrap-up checklist

You should now be able to answer, without the code:

1. What does an LLM compute, and from what quantity are its next-token probabilities derived?
2. Describe the decoding loop in your own words: what is appended, what stops it, and
   what is recomputed at each step.
3. Why does the unembedding use a raw dot product rather than a cosine similarity?
   Where else does the same « multiply, then add » similarity appear in a transformer
   block?
4. In your exercise 3 run, at which layer did « France » take the lead — and what does
   that tell you about how a transformer builds its answer?
